# Employee Data Analysis: Data Kickoff
### Python & Data Engineering · Week 1 · Day 4 · Task 2

## 1. Project title and objective
**Objective:** load the employee dataset, understand its structure and quality
(shape, data types, missing values, duplicates), and answer five business questions
that the available columns can actually answer.

This notebook is the *exploration* version. The same checks live in a reusable script,
`src/analysis.py`, which also writes `reports/data_summary.txt`.

## 2. Dataset description and source
| Item | Value |
|---|---|
| File | `data/employees.csv` |
| Source | Training sample dataset (the same file used in Day 4 Task 1, `task1_terminal_jupyter/project/data/employees.csv`) |
| Rows | 8 employees |
| Columns | `id`, `name`, `department`, `salary`, `city` |
| Format | Comma-separated values with a header row |

> **Note:** this is a very small sample. The results show *how* to analyse the data;
> they are not reliable conclusions about a real company.

## 3. Required Python imports

In [1]:
from pathlib import Path

import pandas as pd

print("pandas version:", pd.__version__)

pandas version: 3.1.1


## 4. Load the dataset
The path is **relative**. The notebook lives in `notebooks/`, so the data is in `../data/`.
If you run the notebook from the project folder instead, the second candidate path is used.
A clear error is raised if the file cannot be found.

In [2]:
candidates = [Path("../data/employees.csv"), Path("data/employees.csv")]
DATA_FILE = next((p for p in candidates if p.is_file()), None)

if DATA_FILE is None:
    raise FileNotFoundError(
        "employees.csv not found. Looked in: "
        + ", ".join(str(p.resolve()) for p in candidates)
        + "\nOpen this notebook from the task2_data_analysis/notebooks folder."
    )

df = pd.read_csv(DATA_FILE)
print(f"Loaded {DATA_FILE} -> {len(df)} rows")

Loaded ..\data\employees.csv -> 8 rows


## 5. First five rows

In [3]:
df.head()

,id,name,department,salary,city
0,1,Arun,Sales,65000,Chennai
1,2,Priya,HR,45000,Bangalore
2,3,Karthik,IT,58000,chennai
3,4,Divya,IT,75000,Mumbai
4,5,Rohan,Finance,81000,CHENNAI


## 6. Shape (rows, columns)

In [4]:
rows, cols = df.shape
print(f"The dataset has {rows} rows and {cols} columns.")
df.shape

The dataset has 8 rows and 5 columns.


(8, 5)

## 7. `info()`: columns, non-null counts and data types

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   id          8 non-null      int64
 1   name        8 non-null      str  
 2   department  8 non-null      str  
 3   salary      8 non-null      int64
 4   city        8 non-null      str  
dtypes: int64(2), str(3)
memory usage: 452.0 bytes


## 8. `describe()`: summary statistics
`include="all"` summarises text columns too (`count`, `unique`, `top`, `freq`), not just numbers.

In [6]:
df.describe(include="all")

,id,name,department,salary,city
count,8.00000,8,8,8.000000,8
unique,NaN,8,4,NaN,6
top,NaN,Arun,IT,NaN,Bangalore
freq,NaN,1,3,NaN,2
mean,4.50000,NaN,NaN,61500.000000,NaN
std,2.44949,NaN,NaN,12950.455039,NaN
min,1.00000,NaN,NaN,45000.000000,NaN
25%,2.75000,NaN,NaN,51000.000000,NaN
50%,4.50000,NaN,NaN,61500.000000,NaN
75%,6.25000,NaN,NaN,69750.000000,NaN


## 9. Missing-value analysis

In [7]:
missing = df.isna().sum()
missing_report = pd.DataFrame({
    "missing_count": missing,
    "missing_percent": (missing / len(df) * 100).round(1),
})
print("Total missing values:", missing.sum())
missing_report

Total missing values: 0


,missing_count,missing_percent
id,0,0.0
name,0,0.0
department,0,0.0
salary,0,0.0
city,0,0.0


## 10. Duplicate-row analysis

In [8]:
print("Fully duplicated rows:", df.duplicated().sum())
print("Duplicated employee ids:", df["id"].duplicated().sum())
print("Duplicated names:", df["name"].duplicated().sum())

Fully duplicated rows: 0
Duplicated employee ids: 0
Duplicated names: 0


## 11. Column data types

In [9]:
print("Column names:", list(df.columns))
df.dtypes.to_frame("dtype")

Column names: ['id', 'name', 'department', 'salary', 'city']


,dtype
id,int64
name,str
department,str
salary,int64
city,str


### 11.1 Data-quality check: city spelling
The same city appears in different letter cases. Pandas treats them as different values,
so the city has to be cleaned before grouping.

In [10]:
print("Raw city values:", sorted(df["city"].unique()))

df["city_clean"] = df["city"].str.strip().str.title()
print("Cleaned city values:", sorted(df["city_clean"].unique()))
print(f"Unique cities: {df['city'].nunique()} raw -> {df['city_clean'].nunique()} cleaned")

Raw city values: ['Bangalore', 'CHENNAI', 'Chennai', 'Hyderabad', 'Mumbai', 'chennai']
Cleaned city values: ['Bangalore', 'Chennai', 'Hyderabad', 'Mumbai']
Unique cities: 6 raw -> 4 cleaned


## 12. Five business questions
Each question uses only the columns in the dataset (`id`, `name`, `department`, `salary`, `city`).
The dataset has no attrition, hire-date or performance columns, so the questions do not cover those topics.

1. What is the average salary across all employees?
2. Which department has the highest average salary?
3. Which city has the highest average salary, and how does salary vary between cities?
4. Which employee has the highest salary?
5. How are employees distributed across departments and cities?

We also check for missing or duplicate employee records (sections 9 and 10).

### Q1. What is the average salary across all employees?

In [11]:
df["salary"].agg(["mean", "median", "min", "max", "std"]).round(2)

mean      61500.00
median    61500.00
min       45000.00
max       81000.00
std       12950.46
Name: salary, dtype: float64

### Q2. Which department has the highest average salary?

In [12]:
by_dept = (df.groupby("department")["salary"]
             .agg(employees="count", avg_salary="mean")
             .sort_values("avg_salary", ascending=False))
by_dept

,employees,avg_salary
department,,
Finance,1,81000.0
IT,3,67000.0
Sales,2,58500.0
HR,2,46500.0


### Q3. Which city has the highest average salary? How does salary vary between cities?

In [13]:
by_city = (df.groupby("city_clean")["salary"]
             .agg(employees="count", avg_salary="mean", min_salary="min", max_salary="max")
             .sort_values("avg_salary", ascending=False))

top_avg = by_city["avg_salary"].max()
top_cities = by_city.index[by_city["avg_salary"] == top_avg].tolist()
print(f"Highest average salary: {top_avg:,.0f} -> {', '.join(top_cities)}")
by_city

Highest average salary: 68,000 -> Chennai, Hyderabad


,employees,avg_salary,min_salary,max_salary
city_clean,,,,
Chennai,3,68000.0,58000,81000
Hyderabad,1,68000.0,68000,68000
Mumbai,2,61500.0,48000,75000
Bangalore,2,48500.0,45000,52000


### Q4. Which employee has the highest salary?

In [14]:
df[df["salary"] == df["salary"].max()][["id", "name", "department", "city_clean", "salary"]]

,id,name,department,city_clean,salary
4,5,Rohan,Finance,Chennai,81000


### Q5. How are employees distributed across departments and cities?

In [15]:
pd.crosstab(df["department"], df["city_clean"], margins=True, margins_name="Total")

city_clean,Bangalore,Chennai,Hyderabad,Mumbai,Total
department,,,,,
Finance,0,1,0,0,1
HR,1,0,0,1,2
IT,0,1,1,1,3
Sales,1,1,0,0,2
Total,2,3,1,2,8


## 13. Data dictionary
Data types are the ones pandas detected above (`df.dtypes`). In pandas 3.x text columns show as `str`;
older pandas versions show them as `object`.

| Column | Detected data type | Description | Example value |
|---|---|---|---|
| `id` | `int64` | Unique employee identifier (1–8, no duplicates) | `1` |
| `name` | `str` | Employee first name | `Arun` |
| `department` | `str` | Department the employee works in (Sales, HR, IT, Finance) | `Sales` |
| `salary` | `int64` | Employee salary as a whole number. The currency and period are not stated (Task 1 assumed INR). | `65000` |
| `city` | `str` | City where the employee is based. Letter case is inconsistent (`Chennai` / `chennai` / `CHENNAI`). | `Chennai` |

The code below builds the same table directly from the data, which shows that the table above matches the real file.

In [16]:
data_dictionary = pd.DataFrame({
    "column": df.columns[:5],
    "detected_dtype": [str(t) for t in df.dtypes.iloc[:5]],
    "example_value": df.iloc[0, :5].tolist(),
    "unique_values": df.iloc[:, :5].nunique().tolist(),
})
data_dictionary

,column,detected_dtype,example_value,unique_values
0,id,int64,1,8
1,name,str,Arun,8
2,department,str,Sales,4
3,salary,int64,65000,8
4,city,str,Chennai,6


## 14. Initial observations and limitations
**Observations**
- 8 rows × 5 columns; **no missing values** and **no duplicate rows or ids**.
- `id` and `salary` are integers; `name`, `department` and `city` are text.
- Average salary is **61,500** (median 61,500), ranging from 45,000 (Priya, HR) to 81,000 (Rohan, Finance).
- **Finance** has the highest department average (81,000) but only **one** employee. IT (3 employees) averages 67,000.
- After cleaning, **Chennai and Hyderabad tie** for the highest city average (68,000). Hyderabad has only one employee.
- The `city` column needs cleaning: 6 raw spellings become 4 real cities.

**Limitations**
- **Very small sample (8 rows).** Group averages are often based on one or two people and are not statistically meaningful.
- The salary currency and period (monthly or annual) are not documented.
- There are no date, attrition, performance or experience columns, so trends and causes cannot be analysed.
- `name` contains first names only and may not be unique in a larger dataset, so `id` should be used as the key.

## 15. Conclusion
The dataset loads cleanly. It is complete (no missing values) and has no duplicate records.
The one data-quality issue is the **inconsistent letter case in `city`**, which is fixed with
`.str.strip().str.title()` before grouping. All five business questions could be answered with the
available columns. Because there are only 8 rows, the answers are a demonstration of the method,
not business conclusions.

**Next steps:** standardise the `city` column at the source, confirm the salary currency and period,
and repeat this analysis on a larger dataset.